### Data Analytics Project - Covid 19 Dataset

In [144]:
# Import libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [145]:
# Import data
df_raw = pd.read_csv('../data/covid_19.csv')
df_raw.head(10)

,country,continent,population,day,time,Cases,Recovered,Deaths,Tests
0,Saint-Helena,Africa,6.115000e+03,2024-06-30,2024-06-30T16:15:16+00:00,2166,2.0,NaN,NaN
1,Falkland-Islands,South-America,3.539000e+03,2024-06-30,2024-06-30T16:15:16+00:00,1930,1930.0,NaN,8632.0
2,Montserrat,North-America,4.965000e+03,2024-06-30,2024-06-30T16:15:16+00:00,1403,1376.0,8.0,17762.0
3,Diamond-Princess,NaN,NaN,2024-06-30,2024-06-30T16:15:16+00:00,712,699.0,13.0,NaN
4,Vatican-City,Europe,7.990000e+02,2024-06-30,2024-06-30T16:15:16+00:00,29,29.0,NaN,NaN
5,Western-Sahara,Africa,6.261610e+05,2024-06-30,2024-06-30T16:15:16+00:00,10,9.0,1.0,NaN
6,MS-Zaandam,NaN,NaN,2024-06-30,2024-06-30T16:15:16+00:00,9,7.0,2.0,NaN
7,China,Asia,1.448471e+09,2024-06-30,2024-06-30T16:15:16+00:00,503302,379053.0,5272.0,160000000.0
8,Tokelau,Oceania,1.378000e+03,2024-06-30,2024-06-30T16:15:16+00:00,80,NaN,NaN,NaN
9,Saint-Pierre-Miquelon,North-America,5.759000e+03,2024-06-30,2024-06-30T16:15:16+00:00,3452,2449.0,2.0,25400.0


In [146]:
# Observe the data
df_raw.isnull().sum()

country        0
continent      2
population     9
day            0
time           0
Cases          0
Recovered     48
Deaths         5
Tests         25
dtype: int64

### We can see that there are multiple missing values on the data. We need to find strategy how to fill.
### My approach:
### 1. Continent and population: I'll remove the rows, since the it is name of a cruise
### 2. Recovered, Deaths and Tests: Median value (for the same region/continent)

In [147]:
# Observe the datatype
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     238 non-null    str    
 1   continent   236 non-null    str    
 2   population  229 non-null    float64
 3   day         238 non-null    str    
 4   time        238 non-null    str    
 5   Cases       238 non-null    int64  
 6   Recovered   190 non-null    float64
 7   Deaths      233 non-null    float64
 8   Tests       213 non-null    float64
dtypes: float64(4), int64(1), str(4)
memory usage: 16.9 KB


In [148]:
# We can change the day and time into string and datetime respectively
df_raw['day'] = pd.to_datetime(df_raw['day']).dt.day_name()
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     238 non-null    str    
 1   continent   236 non-null    str    
 2   population  229 non-null    float64
 3   day         238 non-null    str    
 4   time        238 non-null    str    
 5   Cases       238 non-null    int64  
 6   Recovered   190 non-null    float64
 7   Deaths      233 non-null    float64
 8   Tests       213 non-null    float64
dtypes: float64(4), int64(1), str(4)
memory usage: 16.9 KB


In [149]:
df_raw['time'] = pd.to_datetime(df_raw['time']).dt.normalize()
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   country     238 non-null    str                
 1   continent   236 non-null    str                
 2   population  229 non-null    float64            
 3   day         238 non-null    str                
 4   time        238 non-null    datetime64[us, UTC]
 5   Cases       238 non-null    int64              
 6   Recovered   190 non-null    float64            
 7   Deaths      233 non-null    float64            
 8   Tests       213 non-null    float64            
dtypes: datetime64[us, UTC](1), float64(4), int64(1), str(3)
memory usage: 16.9 KB


In [150]:
# We have removed the cruise
df_raw = df_raw.dropna(subset=['continent'])
df_raw.isnull().sum()

country        0
continent      0
population     7
day            0
time           0
Cases          0
Recovered     48
Deaths         5
Tests         23
dtype: int64

In [151]:
# We can remove the rows with null populations, since the data is not that clear
df_raw.loc[df_raw['population'].isnull()]
df_raw = df_raw.dropna(subset=['population'])

In [152]:
# Now we can fill the missing values on Recovered, Deaths and Tests with the median value of the specific continent or region
df_raw.loc[df_raw['Recovered'].isnull()].groupby('continent').count()

,country,population,day,time,Cases,Recovered,Deaths,Tests
continent,,,,,,,,
Africa,7,7,7,7,7,0,7,6
Asia,11,11,11,11,11,0,11,11
Europe,12,12,12,12,12,0,12,12
North-America,10,10,10,10,10,0,10,10
Oceania,6,6,6,6,6,0,5,2
South-America,2,2,2,2,2,0,2,2


In [153]:
median = df_raw.groupby('continent')['Recovered'].median()
median

continent
Africa             53569.0
Asia              727915.0
Europe           2004268.0
North-America      29805.0
Oceania             9563.0
South-America    1105645.0
Name: Recovered, dtype: float64

In [155]:
# Fill the missing values
df_raw['Recovered'] =  df_raw['Recovered'].fillna(
    df_raw['continent'].map(median)
)

country        0
continent      0
population     0
day            0
time           0
Cases          0
Recovered      0
Deaths         5
Tests         16
dtype: int64

In [157]:
# Do the same thing with deaths and tests
death_median = df_raw.groupby('continent')['Deaths'].median()
death_median
df_raw['Deaths'] = df_raw['Deaths'].fillna(
    df_raw['continent'].map(death_median)
)

In [158]:
test_median =  df_raw.groupby('continent')['Tests'].median()
df_raw['Tests'] = df_raw['Tests'].fillna(
    df_raw['continent'].map(test_median)
)

In [164]:
df_raw.columns = df_raw.columns.str.lower()
df_raw.columns

Index(['country', 'continent', 'population', 'day', 'time', 'cases',
       'recovered', 'deaths', 'tests'],
      dtype='str')

In [165]:
# Now we already have clean data can import to SQL for query or doing data exploration in python
df = df_raw.copy()

In [169]:
# Quick Statistical Insight
df.describe()

,population,cases,recovered,deaths,tests
count,2.290000e+02,2.290000e+02,2.290000e+02,2.290000e+02,2.290000e+02
mean,3.469404e+07,3.077525e+06,2.600729e+06,3.076972e+04,3.085712e+07
std,1.386374e+08,1.006101e+07,9.205055e+06,1.096416e+05,1.158633e+08
min,7.990000e+02,1.000000e+01,2.000000e+00,1.000000e+00,7.850000e+03
25%,4.454310e+05,2.733400e+04,2.980500e+04,2.250000e+02,3.059410e+05
50%,5.797805e+06,2.099060e+05,2.110800e+05,2.250000e+03,1.941032e+06
75%,2.210284e+07,1.356546e+06,1.503989e+06,1.492400e+04,1.171151e+07
max,1.448471e+09,1.118201e+08,1.098144e+08,1.219487e+06,1.186852e+09


In [195]:
# Aggregation by Continent
aggregate_by_cont = df.groupby('continent')[['cases', 'recovered', 'deaths', 'tests']].sum()
# the most cases?
aggregate_by_cont.sort_values('cases', ascending=False).head(1)

,cases,recovered,deaths,tests
continent,,,,
Europe,253406198,259848390.0,2114042.0,2.859441e+09


In [196]:
# the most deaths
aggregate_by_cont.sort_values('deaths', ascending=False).head(1)  

,cases,recovered,deaths,tests
continent,,,,
Europe,253406198,259848390.0,2114042.0,2.859441e+09


In [198]:
# most test conducted
aggregate_by_cont.sort_values('tests', ascending=False).head(1)

,cases,recovered,deaths,tests
continent,,,,
Europe,253406198,259848390.0,2114042.0,2.859441e+09


In [ ]:
# Per capita metrics - we add new columns
df['cases_per_100k'] = df['cases']/df['population']*100000
df['deaths_per_100k'] = df['deaths']/df['population']*100000

In [212]:
# Top 10 countries with highest cases
df.sort_values('cases_per_100k', ascending=False).head(10)

,country,continent,population,day,time,cases,recovered,deaths,tests,cases_per_100k,deaths_per_100k
152,Brunei,Asia,445431.0,Sunday,2024-06-30 00:00:00+00:00,343719,243601.0,225.0,717784.0,77165.486910,50.512874
87,San-Marino,Europe,34085.0,Sunday,2024-06-30 00:00:00+00:00,26185,26011.0,128.0,196855.0,76822.649259,375.531759
81,Faeroe-Islands,Europe,49233.0,Sunday,2024-06-30 00:00:00+00:00,34658,2004268.0,28.0,778000.0,70395.872687,56.872423
224,S-Korea,Asia,51329899.0,Sunday,2024-06-30 00:00:00+00:00,34571873,34535939.0,35934.0,15804065.0,67352.310590,70.005982
172,Austria,Europe,9066710.0,Sunday,2024-06-30 00:00:00+00:00,6081287,6054934.0,22542.0,211273524.0,67072.697814,248.623812
11,Niue,Oceania,1622.0,Sunday,2024-06-30 00:00:00+00:00,1059,1056.0,27.5,98964.0,65289.765721,1695.437731
201,Slovenia,Europe,2078034.0,Sunday,2024-06-30 00:00:00+00:00,1356546,1349424.0,7100.0,2847701.0,65280.260092,341.669097
73,Andorra,Europe,77463.0,Sunday,2024-06-30 00:00:00+00:00,48015,2004268.0,165.0,249838.0,61984.431277,213.004918
161,Martinique,North-America,374087.0,Sunday,2024-06-30 00:00:00+00:00,230354,29805.0,1102.0,828928.0,61577.654396,294.583880
221,France,Europe,65584518.0,Sunday,2024-06-30 00:00:00+00:00,40138560,39970918.0,167642.0,271490188.0,61201.273142,255.612155


In [213]:
# Top 10 contries with highest deaths
df.sort_values('deaths_per_100k', ascending=False).head(10)

,country,continent,population,day,time,cases,recovered,deaths,tests,cases_per_100k,deaths_per_100k
4,Vatican-City,Europe,799.0,Sunday,2024-06-30 00:00:00+00:00,29,29.0,12218.0,11394556.0,3629.536921,1.529161e+06
1,Falkland-Islands,South-America,3539.0,Sunday,2024-06-30 00:00:00+00:00,1930,1930.0,22407.0,8632.0,54535.179429,6.331450e+05
0,Saint-Helena,Africa,6115.0,Sunday,2024-06-30 00:00:00+00:00,2166,2.0,921.0,804909.0,35421.095666,1.506132e+04
8,Tokelau,Oceania,1378.0,Sunday,2024-06-30 00:00:00+00:00,80,9563.0,27.5,98964.0,5805.515239,1.995646e+03
11,Niue,Oceania,1622.0,Sunday,2024-06-30 00:00:00+00:00,1059,1056.0,27.5,98964.0,65289.765721,1.695438e+03
181,Peru,South-America,33684208.0,Sunday,2024-06-30 00:00:00+00:00,4572667,4350506.0,222161.0,39010194.0,13575.106174,6.595405e+02
202,Bulgaria,Europe,6844597.0,Sunday,2024-06-30 00:00:00+00:00,1339851,1292944.0,38748.0,11671043.0,19575.308817,5.661108e+02
192,Hungary,Europe,9606259.0,Sunday,2024-06-30 00:00:00+00:00,2230232,2152155.0,49048.0,11394556.0,23216.446694,5.105838e+02
146,Bosnia-and-Herzegovina,Europe,3249317.0,Sunday,2024-06-30 00:00:00+00:00,403615,379084.0,16388.0,1884721.0,12421.533510,5.043521e+02
149,North-Macedonia,Europe,2081304.0,Sunday,2024-06-30 00:00:00+00:00,350567,337068.0,9976.0,2226216.0,16843.623036,4.793149e+02


In [ ]:
# Recovery rate
df['recovery_rate'] = df['recovered']/df['cases']*100

In [215]:
# Top 10 countries with highest recovery rate
df.sort_values('recovery_rate', ascending=False).head(10)

,country,continent,population,day,time,cases,recovered,deaths,tests,cases_per_100k,deaths_per_100k,recovery_rate
8,Tokelau,Oceania,1378.0,Sunday,2024-06-30 00:00:00+00:00,80,9563.0,27.5,98964.0,5805.515239,1995.645864,11953.750000
39,Monaco,Europe,39783.0,Sunday,2024-06-30 00:00:00+00:00,17181,2004268.0,67.0,78646.0,43186.788327,168.413644,11665.607357
37,Gibraltar,Europe,33704.0,Sunday,2024-06-30 00:00:00+00:00,20550,2004268.0,113.0,534283.0,60971.991455,335.271778,9753.128954
36,Liechtenstein,Europe,38387.0,Sunday,2024-06-30 00:00:00+00:00,21574,2004268.0,94.0,112457.0,56201.318155,244.874567,9290.201168
81,Faeroe-Islands,Europe,49233.0,Sunday,2024-06-30 00:00:00+00:00,34658,2004268.0,28.0,778000.0,70395.872687,56.872423,5782.988055
57,Isle-of-Man,Europe,85732.0,Sunday,2024-06-30 00:00:00+00:00,38008,2004268.0,116.0,150753.0,44333.504409,135.305370,5273.279310
73,Andorra,Europe,77463.0,Sunday,2024-06-30 00:00:00+00:00,48015,2004268.0,165.0,249838.0,61984.431277,213.004918,4174.253879
93,Suriname,South-America,596831.0,Sunday,2024-06-30 00:00:00+00:00,82588,1105645.0,1408.0,242207.0,13837.753066,235.912679,1338.747760
163,Iceland,Europe,345393.0,Sunday,2024-06-30 00:00:00+00:00,209906,2004268.0,229.0,1996384.0,60773.090364,66.301286,954.840738
55,Anguilla,North-America,15230.0,Sunday,2024-06-30 00:00:00+00:00,3904,29805.0,12.0,51382.0,25633.617859,78.791858,763.447746


### I think the something is off from the dataset, if we observed the numbers of cases compared to recovered, there are discrepancies among them.

In [216]:
df.isnull().sum()

country            0
continent          0
population         0
day                0
time               0
cases              0
recovered          0
deaths             0
tests              0
cases_per_100k     0
deaths_per_100k    0
recovery_rate      0
dtype: int64